# Programação para Ciência e Engenharia de Dados
## Aula prática — NumPy e pandas

Fluxo:

`dados brutos → exploração → limpeza → transformação → integração → análise → interpretação → exportação`

Use este notebook em paralelo aos slides. As células estão organizadas para demonstração, discussão e pequenas práticas.


## 0. Preparação


In [ ]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)


## 1. NumPy — vetorização


In [ ]:
valores = [10, 20, 30, 40]
resultado = []
for valor in valores:
    resultado.append(valor * 1.10)
resultado


In [ ]:
valores_np = np.array([10, 20, 30, 40])
valores_np * 1.10


In [ ]:
dados = np.array([10, 50, 120, 30])
print("Média:", dados.mean())
print("Soma:", dados.sum())
print("Máximo:", dados.max())
print("Filtro > 40:", dados[dados > 40])


### Prática rápida
Calcule média, máximo, mínimo, valores acima de 30°C e quantidade de valores acima da média.


In [ ]:
temperaturas = np.array([21.5, 28.2, 31.7, 19.8, 35.1, 27.4])

# Resolva aqui


### Solução


In [ ]:
media = temperaturas.mean()
print("Média:", media)
print("Máxima:", temperaturas.max())
print("Mínima:", temperaturas.min())
print("Acima de 30:", temperaturas[temperaturas > 30])
print("Acima da média:", (temperaturas > media).sum())


## 2. Carregando os dados


In [ ]:
# No Colab, faça upload dos arquivos operacoes.csv e equipamentos.csv
from google.colab import drive
drive.mount('/content/drive')
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/base/operacoes.csv")
df.head()

## 3. Exploração — entender antes de alterar


In [ ]:
print("Dimensão:", df.shape)
print("\nColunas:")
print(df.columns)
print("\nTipos:")
print(df.dtypes)


In [ ]:
df.info()


In [ ]:
df.describe(include="all")


In [ ]:
print("Valores ausentes:")
print(df.isna().sum())

print("\nDuplicados:", df.duplicated().sum())


### Discussão
- O que uma linha representa?
- Qual é a granularidade?
- Quais colunas deveriam ser numéricas?
- Quais problemas já conseguimos perceber?


## 4. Seleção, filtros e ordenação


In [ ]:
df[["equipamento", "horas_trabalhadas", "combustivel_litros"]].head()


In [ ]:
df.loc[0:4, ["equipamento", "horas_trabalhadas"]]


In [ ]:
df.iloc[0:5, 0:3]


In [ ]:
df["horas_num"] = pd.to_numeric(df["horas_trabalhadas"], errors="coerce")
df[df["horas_num"] > 8]


In [ ]:
df[
    (df["horas_num"] > 8) &
    (df["combustivel_litros"] > 100)
]


In [ ]:
df.sort_values("combustivel_litros", ascending=False).head()


## 5. Limpeza


In [ ]:
df_limpo = df.copy()

df_limpo["equipamento"] = (
    df_limpo["equipamento"]
      .str.strip()
      .str.upper()
      .str.replace(" ", "-", regex=False)
)

df_limpo["obra"] = (
    df_limpo["obra"]
      .str.strip()
      .str.title()
)

df_limpo[["equipamento", "obra"]].head(10)


In [ ]:
df_limpo["horas_trabalhadas"] = pd.to_numeric(
    df_limpo["horas_trabalhadas"],
    errors="coerce"
)

df_limpo["data"] = pd.to_datetime(
    df_limpo["data"],
    format="%d/%m/%Y",
    errors="coerce"
)

df_limpo.isna().sum()


In [ ]:
df_limpo[df_limpo["horas_trabalhadas"] < 0]


In [ ]:
df_limpo.loc[
    df_limpo["horas_trabalhadas"] < 0,
    "horas_trabalhadas"
] = np.nan

df_limpo = df_limpo.drop_duplicates()

print("Duplicados:", df_limpo.duplicated().sum())
print(df_limpo.isna().sum())


### Discussão
Evite corrigir silenciosamente. Um valor `-4` poderia ser um erro de digitação, mas transformá-lo automaticamente em `4` seria uma suposição.


## 6. Transformação


In [ ]:
df_limpo["consumo_hora"] = (
    df_limpo["combustivel_litros"] /
    df_limpo["horas_trabalhadas"]
)

df_limpo["produtividade"] = (
    df_limpo["producao_m3"] /
    df_limpo["horas_trabalhadas"]
)

df_limpo.loc[
    df_limpo["horas_trabalhadas"] <= 0,
    ["consumo_hora", "produtividade"]
] = np.nan

df_limpo["mes"] = df_limpo["data"].dt.month

df_limpo["utilizacao"] = np.where(
    df_limpo["horas_trabalhadas"] > 8,
    "ALTA UTILIZACAO",
    "NORMAL"
)

df_limpo.head()


## 7. Agrupamento e agregação


In [ ]:
df_limpo.groupby("equipamento")["horas_trabalhadas"].sum()


In [ ]:
resumo = (
    df_limpo.groupby("equipamento")
      .agg(
          horas=("horas_trabalhadas", "sum"),
          combustivel=("combustivel_litros", "sum"),
          producao=("producao_m3", "sum"),
          consumo_medio=("consumo_hora", "mean"),
          produtividade_media=("produtividade", "mean"),
          registros=("equipamento", "size")
      )
      .reset_index()
)

resumo


In [ ]:
df_limpo.groupby("obra").agg(
    horas=("horas_trabalhadas", "sum"),
    producao=("producao_m3", "sum")
).reset_index()


## 8. Integração com outro DataFrame


In [ ]:
equipamentos = pd.read_csv("equipamentos.csv")
equipamentos


In [ ]:
df_integrado = df_limpo.merge(
    equipamentos,
    on="equipamento",
    how="left"
)

print("Antes:", len(df_limpo))
print("Depois:", len(df_integrado))
df_integrado.head()


In [ ]:
print("Chave única no cadastro:", equipamentos["equipamento"].is_unique)

print("\nOperações sem cadastro:")
display(df_integrado[df_integrado["tipo"].isna()])


## 9. Análise — responder perguntas


In [ ]:
# Qual equipamento trabalhou mais?
df_integrado.groupby("equipamento")["horas_trabalhadas"].sum().sort_values(ascending=False)


In [ ]:
# Qual equipamento consumiu mais?
df_integrado.groupby("equipamento")["combustivel_litros"].sum().sort_values(ascending=False)


In [ ]:
# Qual obra teve maior produção?
df_integrado.groupby("obra")["producao_m3"].sum().sort_values(ascending=False)


In [ ]:
# Qual tipo possui maior produtividade média?
df_integrado.groupby("tipo")["produtividade"].mean().sort_values(ascending=False)


In [ ]:
# Maiores consumos por hora
df_integrado[
    ["equipamento", "data", "horas_trabalhadas", "combustivel_litros", "consumo_hora"]
].sort_values("consumo_hora", ascending=False).head()


## 10. Interpretação
Resultado calculado e conclusão causal são coisas diferentes.

**Podemos dizer:** `ESC-03 apresentou maior consumo por hora no dataset`.

**Não podemos concluir automaticamente:** `ESC-03 está com defeito`.

Para isso, seriam necessárias outras evidências.


In [ ]:
df_integrado["consumo_hora"].describe()


## 11. Organizando em funções


In [ ]:
def carregar_dados(caminho):
    return pd.read_csv(caminho)

def limpar_dados(df):
    df = df.copy()
    df["equipamento"] = (
        df["equipamento"].str.strip().str.upper().str.replace(" ", "-", regex=False)
    )
    df["obra"] = df["obra"].str.strip().str.title()
    df["horas_trabalhadas"] = pd.to_numeric(df["horas_trabalhadas"], errors="coerce")
    df["data"] = pd.to_datetime(df["data"], format="%d/%m/%Y", errors="coerce")
    df.loc[df["horas_trabalhadas"] < 0, "horas_trabalhadas"] = np.nan
    return df.drop_duplicates()

def transformar_dados(df):
    df = df.copy()
    df["consumo_hora"] = df["combustivel_litros"] / df["horas_trabalhadas"]
    df["produtividade"] = df["producao_m3"] / df["horas_trabalhadas"]
    df.loc[df["horas_trabalhadas"] <= 0, ["consumo_hora", "produtividade"]] = np.nan
    df["utilizacao"] = np.where(df["horas_trabalhadas"] > 8, "ALTA UTILIZACAO", "NORMAL")
    return df

def gerar_resumo(df):
    return (
        df.groupby("equipamento")
          .agg(
              horas=("horas_trabalhadas", "sum"),
              combustivel=("combustivel_litros", "sum"),
              producao=("producao_m3", "sum"),
              consumo_medio=("consumo_hora", "mean"),
              produtividade_media=("produtividade", "mean")
          )
          .reset_index()
    )


In [ ]:
dados = carregar_dados("operacoes.csv")
dados = limpar_dados(dados)
dados = transformar_dados(dados)

dados = dados.merge(
    equipamentos,
    on="equipamento",
    how="left"
)

resumo_final = gerar_resumo(dados)
resumo_final


## 12. Exportação


In [ ]:
dados.to_csv("operacoes_tratadas.csv", index=False)
resumo_final.to_csv("resumo_equipamentos.csv", index=False)

print("Arquivos gerados.")


## 13. Desafio final para os alunos

Resolva sem copiar as células anteriores:

1. Qual fabricante acumulou mais horas trabalhadas?
2. Qual obra apresentou maior produtividade média?
3. Qual equipamento possui maior consumo médio por hora?
4. Quantos valores ausentes restaram após a limpeza?
5. Quantas operações foram classificadas como `ALTA UTILIZACAO`?
6. Existe algum comportamento que mereça investigação?
7. O que é possível afirmar com os dados?
8. O que não é possível afirmar?


In [ ]:
# Resolução dos alunos


# Síntese

```text
DADOS
 ↓
EXPLORAÇÃO
 ↓
LIMPEZA
 ↓
TRANSFORMAÇÃO
 ↓
AGREGAÇÃO
 ↓
INTEGRAÇÃO
 ↓
ANÁLISE
 ↓
INTERPRETAÇÃO
 ↓
EXPORTAÇÃO
```

O objetivo não é memorizar métodos do pandas, mas aprender a escolher operações coerentes com o problema e com o significado dos dados.
